In [3]:
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset,DataLoader
import torch.optim as optim

In [4]:
if (torch.cuda.is_available()):
    print(torch.cuda.get_device_name(0))
else:
    print("CPU");

NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
from torchvision import transforms,datasets

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder("data/train",transform=transform)
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True,num_workers=2)

test_dataset =  datasets.ImageFolder("data/test",transform=test_transform)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True,num_workers=2)


In [18]:
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16,128),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            nn.Linear(128,32),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(32,2)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x


In [19]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = MyCNN().to(device)
print(list(model.parameters()))


cuda
[Parameter containing:
tensor([[[[-1.8524e-01, -5.8681e-02,  1.2071e-01],
          [ 9.9654e-02, -5.1766e-02,  6.0823e-02],
          [-1.8615e-02,  4.9586e-02,  6.5148e-05]],

         [[-1.1959e-01,  1.8120e-01, -1.3514e-01],
          [ 1.0739e-01, -3.6542e-02,  9.0745e-02],
          [-9.5814e-02,  9.4771e-02,  7.1743e-02]],

         [[ 1.6284e-01, -1.7353e-01,  1.1311e-01],
          [ 1.7138e-01, -2.7467e-02, -9.1148e-02],
          [ 1.3608e-01, -1.3844e-01, -1.8574e-03]]],


        [[[ 9.6776e-02, -4.9992e-02, -1.0683e-01],
          [ 1.1964e-01, -1.5641e-01, -1.8267e-01],
          [ 3.3807e-02,  8.4195e-02, -1.4151e-01]],

         [[ 2.5439e-02,  5.6199e-02, -1.4127e-01],
          [ 1.5493e-01, -6.7208e-02,  5.7919e-02],
          [-7.7898e-02, -6.9635e-02, -2.0795e-02]],

         [[-1.2124e-01, -1.0523e-01, -1.7195e-01],
          [-1.3500e-01,  4.0586e-02,  3.9558e-02],
          [-1.7805e-01, -6.2967e-02,  1.1134e-01]]],


        [[[-3.5241e-02, -4.0204e-02, -

In [20]:
epochs = 20
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = 0.001)

In [16]:
for images, labels in train_loader:
    images = images.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

    print("Pred:", predicted[:10].cpu().numpy())
    print("Labels:", labels[:10].numpy())
    break

Pred: [0 0 0 1 1 0 1 0 1 0]
Labels: [0 0 1 0 0 1 1 1 0 0]


In [21]:
for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for images,labels in train_loader:
        images,labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")
    accuracy = 100 * correct / total
    print(f"Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")


Epoch [1/20], Loss: 0.6492
Loss: 0.6492, Accuracy: 62.78%
Epoch [2/20], Loss: 0.5399
Loss: 0.5399, Accuracy: 72.77%
Epoch [3/20], Loss: 0.4902
Loss: 0.4902, Accuracy: 76.91%
Epoch [4/20], Loss: 0.4403
Loss: 0.4403, Accuracy: 79.70%
Epoch [5/20], Loss: 0.4106
Loss: 0.4106, Accuracy: 81.67%
Epoch [6/20], Loss: 0.3846
Loss: 0.3846, Accuracy: 83.02%
Epoch [7/20], Loss: 0.3655
Loss: 0.3655, Accuracy: 83.83%
Epoch [8/20], Loss: 0.3508
Loss: 0.3508, Accuracy: 84.84%
Epoch [9/20], Loss: 0.3263
Loss: 0.3263, Accuracy: 86.03%
Epoch [10/20], Loss: 0.3135
Loss: 0.3135, Accuracy: 86.59%
Epoch [11/20], Loss: 0.3038
Loss: 0.3038, Accuracy: 87.00%
Epoch [12/20], Loss: 0.2893
Loss: 0.2893, Accuracy: 87.83%
Epoch [13/20], Loss: 0.2728
Loss: 0.2728, Accuracy: 88.44%
Epoch [14/20], Loss: 0.2587
Loss: 0.2587, Accuracy: 89.38%
Epoch [15/20], Loss: 0.2509
Loss: 0.2509, Accuracy: 89.56%
Epoch [16/20], Loss: 0.2369
Loss: 0.2369, Accuracy: 89.94%
Epoch [17/20], Loss: 0.2284
Loss: 0.2284, Accuracy: 90.47%
Epoch 

In [22]:
model.eval()

MyCNN(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplac

In [23]:
correct = 0
total = 0

test_loss = 0

with torch.no_grad():
    for images,labels in test_loader:
        images,labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs,labels)

        test_loss += loss.item()

        _,predicted = torch.max(outputs,1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    test_accuracy = 100 * correct / total
    avg_test_loss = test_loss / len(test_loader)

    print(f"Test Loss: {avg_test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.2f}%")


Test Loss: 0.2428
Test Accuracy: 89.80%
